In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
os.chdir("/Users/kanhapandey/Projects/SIH_PAIMANA")

In [ ]:
df = pd.read_csv("data/raw/quaterly_frozen_or_deleted_projects.csv")

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.drop(13, inplace=True)
df

In [ ]:
df.drop("Sl. No.", axis=1, inplace=True)
df

In [ ]:
def split_name_agency_code_state(text: str):
    if pd.isna(text):
        return pd.Series([None, None, None, None])

    text = text.replace('\n', ' ').strip()

    # -------------------------
    # Getting state name
    # -------------------------
    s_close_idx = text.rfind(')')
    s_balance = 0
    s_open_idx = None

    for i in range(s_close_idx, -1, -1):
        s_ch = text[i]

        if s_ch == ')':
            s_balance += 1
        elif s_ch == '(':
            s_balance -= 1

            if s_balance == 0:
                s_open_idx = i
                break

    state_name = text[s_open_idx + 1:s_close_idx].strip()

    # Remaining text contains project name + agency + code
    remainder = text[:s_open_idx].strip()

    # -------------------------
    # Getting project code
    # -------------------------
    c_close_idx = remainder.rfind(')')
    c_balance = 0
    c_open_idx = None

    for i in range(c_close_idx, -1, -1):
        c_ch = remainder[i]

        if c_ch == ')':
            c_balance += 1
        elif c_ch == '(':
            c_balance -= 1

            if c_balance == 0:
                c_open_idx = i
                break

    code = remainder[c_open_idx + 1:c_close_idx].strip()

    # Remaining text contains project name + agency
    name_agency = remainder[:c_open_idx].strip()

    # -------------------------
    # Getting agency name
    # -------------------------
    a_close_idx = name_agency.rfind(')')
    a_balance = 0
    a_open_idx = None

    for i in range(a_close_idx, -1, -1):
        a_ch = name_agency[i]

        if a_ch == ')':
            a_balance += 1
        elif a_ch == '(':
            a_balance -= 1

            if a_balance == 0:
                a_open_idx = i
                break

    agency = name_agency[a_open_idx + 1:a_close_idx].strip()

    # Remaining text is project name
    project_name = name_agency[:a_open_idx].strip()

    return pd.Series([project_name, agency, code, state_name])

In [ ]:
df[["project_name", "agency", "project_code", "state"]] = (
    df["Project Name (Agency Name) (Project Code) (State Name)"].apply(split_name_agency_code_state)
)
df.head()

In [ ]:
df.drop("Project Name (Agency Name) (Project Code) (State Name)", axis=1, inplace=True)
df.head()

In [ ]:
df["status"]="Frozen or Deleted"
df.head()

In [ ]:
missing_tokens = {'-', 'NA', 'na', 'N/A', ''}
df["Cumulative Expenditure in Rs. Crore"] = (df["Cumulative Expenditure in Rs. Crore"].replace(missing_tokens, np.nan).str.replace(",", "", regex=False))
df["Cumulative Expenditure in Rs. Crore"] = pd.to_numeric(df["Cumulative Expenditure in Rs. Crore"], errors="coerce")

In [ ]:
df["Date of Commissioning Original (MM/YYYY)"] = df["Date of Commissioning Original (MM/YYYY)"].replace(missing_tokens, np.nan)
df["Date of Commissioning Original (MM/YYYY)"] = pd.to_datetime(df["Date of Commissioning Original (MM/YYYY)"], format='%m/%Y', errors='coerce')

In [ ]:
df.info()

In [ ]:
df.columns = df.columns.str.lower()
print(df.columns)

In [ ]:
df["physical progress (in percentage)"] = 0.0
df["physical progress (in percentage)"] = df["physical progress (in percentage)"].astype(float)

In [ ]:
df.info()

In [ ]:
df["original_cost_cr"] = df["original cost in rs. crore"].replace(missing_tokens, np.nan).str.replace(",", "", regex=False).astype(float)
df["date_of_approval"] = df["date of commissioning original (mm/yyyy)"]
df.drop("original cost in rs. crore", axis=1, inplace=True)
df.drop("date of commissioning original (mm/yyyy)", axis=1, inplace=True)

In [ ]:
df.info()

In [ ]:
df['start_date'] = pd.Series(dtype="datetime64[us]")
df['actual_doc'] = pd.Series(dtype="datetime64[us]")
df['target_doc'] = pd.Series(dtype="datetime64[us]")
df['revised_doc'] = pd.Series(dtype="datetime64[us]")
df["revised_cost_cr"] = pd.Series(dtype="float")
df["ministry"] = pd.Series(dtype="string")
df.info()

In [ ]:
final_ordered_cols = [
    'project_code', 'project_name', 'agency', 'state', 'date_of_approval',
       'start_date', 'actual_doc', 'target_doc', 'revised_doc',
       'original_cost_cr', 'revised_cost_cr',
       'cumulative expenditure in rs. crore',
       'physical progress (in percentage)', 'ministry', 'sector', 'status',
]
df = df[final_ordered_cols]
df.head()

In [ ]:
df.to_csv("data/cleaned/quaterly_frozen_or_deleted_projects_clean.csv", index=False)

In [ ]:
df.info()